# Train TinyLlama HelpSteer2 Adapters

This notebook trains five independent TinyLlama LoRA specialists for helpfulness, correctness, coherence, complexity, and verbosity. It uses the central experiment configuration and provides graceful stop controls for long Colab runs.

## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Check the GPU

In [ ]:
!nvidia-smi

## 3. Install dependencies

The pandas and NumPy versions are pinned for compatibility with the standard Colab environment. TinyLlama uses standard LoRA training without TorchAO or 4-bit quantization. Colab may include an old optional `torchao` package that is incompatible with current Transformers, so the install cell removes it.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "transformers<5" datasets peft accelerate pyyaml tensorboard

Restart the runtime once after installation so Python forgets any previously imported TorchAO or PyTorch modules. Then rerun the repository cell and continue with the GPU and validation cells; you do not need to reinstall the dependencies again in the same Colab session.

## 4. Show important files

In [ ]:
!pwd
!ls
!ls configs
!ls scripts
!ls src

## 5. Validate the config and inspect HelpSteer2

In [ ]:
!python scripts/validate_tinyllama_helpsteer2_config.py
!python scripts/inspect_helpsteer2_dataset.py --split "train[:10000]"

## 6. Start TensorBoard

Open the **Scalars** view to inspect training and evaluation loss against `global_step`. New points appear as training writes event logs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tensorboard/tinyllama_helpsteer2

## 7. Manual one-adapter-at-a-time training

Each code block below starts exactly one adapter as a background process, so the stop and status cells remain usable. Start the next adapter manually only after the status cell shows that training has finished and you have checked the previous adapter's `train_loss` and `eval_loss` in TensorBoard. The first line of each training cell removes stale stop files from an earlier run.

### Helpfulness

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes helpfulness \
  --split "train[:10000]" \
  --eval_split "train[10000:11000]" \
  --num_epochs 50 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes helpfulness

### Correctness

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes correctness \
  --split "train[:10000]" \
  --eval_split "train[10000:11000]" \
  --num_epochs 50 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes correctness

### Coherence

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes coherence \
  --split "train[:10000]" \
  --eval_split "train[10000:11000]" \
  --num_epochs 50 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes coherence

### Complexity

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes complexity \
  --split "train[:10000]" \
  --eval_split "train[10000:11000]" \
  --num_epochs 50 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes complexity

### Verbosity

In [ ]:
!rm -f STOP_CURRENT_ADAPTER STOP_TRAINING
!nohup python -u scripts/train_tinyllama_helpsteer2_adapters.py \
  --attributes verbosity \
  --split "train[:10000]" \
  --eval_split "train[10000:11000]" \
  --num_epochs 50 \
  --batch_size 8 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 500 \
  --use_tensorboard > /content/tinyllama_helpsteer2_training.log 2>&1 &

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py --attributes verbosity

## 8. Stop the current adapter

Run this cell while background training is active if you want to stop the current adapter. The script does not wait until the end of the epoch. It stops after the next save step, saves the adapter, removes `STOP_CURRENT_ADAPTER`, and exits gracefully. With `--save_steps 500`, the maximum wait is roughly until the next 500-step checkpoint.

In [ ]:
!touch STOP_CURRENT_ADAPTER

## 9. Check the running training

Run this cell at any time to see whether an adapter process is active and to inspect the latest training output.

In [ ]:
!pgrep -af "[t]rain_tinyllama_helpsteer2_adapters.py" || echo "No adapter training process is running."
!tail -n 50 /content/tinyllama_helpsteer2_training.log 2>/dev/null || echo "No training log exists yet."

## 10. Inspect saved logs

Use these folders to inspect the CSV logs and TensorBoard event files.

In [ ]:
!ls results/tinyllama_helpsteer2_training_logs
!ls results/tensorboard/tinyllama_helpsteer2

## 11. Export training curves

In [ ]:
!python scripts/export_tinyllama_training_curves.py || true

## 12. Final adapter check

In [ ]:
!python scripts/check_tinyllama_helpsteer2_adapters.py

## 13. Create a local adapter backup

Run this only after all five adapters are finished. The zip file is a generated local backup and must stay out of Git.

In [ ]:
!zip -r tinyllama_helpsteer2_adapters.zip adapters/tinyllama-helpsteer2-*-adapter/

## 14. Git safety check

Adapters, checkpoints, safetensors, `.bin` files, model weights, TensorBoard events, and zip files are generated artifacts. Keep them out of Git unless a small result file is intentionally selected later.

In [ ]:
!git status